# Notebook 3 — Interactive map of the frozen final pipeline

This notebook loads the pipeline selected in Notebook 2 from
`metrics/final_selection.json`: the chosen CNN run, optional second-stage
XGBoost model, and the threshold fitted on validation data. It does **not**
choose a model or threshold, and it must not be used to tune either.

Predictions are written as one GeoJSON per city/date/frozen-pipeline. The
map's satellite basemap is for orientation only; it is not necessarily
date-matched model evidence.


## 1. Setup — select a completed Notebook 2 experiment


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import sys
BASE = "/content/drive/MyDrive/War-Damage-Detection"
if BASE not in sys.path:
    sys.path.append(BASE)

import json
import os
import numpy as np
import geopandas as gpd
import torch

from pipeline import (
    CITY_REGISTRY, EXPERIMENTS_DIR, load_city, city_source,
    label_matrix, propagate_labels, split_assignment, strip_dates,
    temporal_offsets, spatial_smooth, neighbour_features, temporal_features,
    build_model, predict_probs, experiment_dirs,
)

# Must name an experiment that has completed Notebook 2 model selection.
EXPERIMENT_NAME = "exp001_tiny_cnn_sar_temporal"

print("experiments on Drive:", sorted(os.listdir(EXPERIMENTS_DIR)))
EXP_ROOT, OUT = experiment_dirs(EXPERIMENT_NAME)

selection_path = os.path.join(OUT["metrics"], "final_selection.json")
if not os.path.exists(selection_path):
    raise FileNotFoundError(
        f"{selection_path} is missing. Run Notebook 2 through validation "
        "model selection before using this notebook.")
with open(selection_path) as fh:
    FINAL_SELECTION = json.load(fh)
if FINAL_SELECTION.get("experiment_name") != EXPERIMENT_NAME:
    raise RuntimeError("final_selection.json belongs to another experiment")

FINAL_TAG = FINAL_SELECTION["run"]
FINAL_VARIANT = FINAL_SELECTION["variant"]
FINAL_THRESHOLD = float(FINAL_SELECTION["threshold"])
checkpoint_path = os.path.join(OUT["models"], f"{FINAL_TAG}.pt")
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"selected checkpoint is missing: {checkpoint_path}")

ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
if not ckpt.get("complete"):
    raise RuntimeError(f"{checkpoint_path} is not a completed model checkpoint")
cfg = ckpt["config"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(ckpt["model"], ckpt["channel_names"], device=device,
                    quiet=True, **ckpt["params"])
model.load_state_dict(ckpt["state_dict"])
model.eval()

STACKER = None
if FINAL_VARIANT.startswith("xgb_"):
    stacker_path = os.path.join(OUT["models"], "stackers.pt")
    if not os.path.exists(stacker_path):
        raise FileNotFoundError(
            f"{stacker_path} is required by selected variant {FINAL_VARIANT}")
    saved = torch.load(stacker_path, map_location="cpu", weights_only=False)
    if saved.get("format") != 1 or saved.get("config") != cfg:
        raise RuntimeError("stackers.pt does not match the selected checkpoint config")
    try:
        STACKER = saved["stackers"][FINAL_TAG][FINAL_VARIANT]
    except KeyError as exc:
        raise RuntimeError("selected XGBoost stacker is missing from stackers.pt") from exc

print(f"loaded final pipeline: {FINAL_TAG} / {FINAL_VARIANT}")
print(f"validation-fitted threshold: {FINAL_THRESHOLD:.4f}")
print(f"CNN: {ckpt['model']}, best epoch {ckpt['best_epoch']}, "
      f"validation PR-AUC {ckpt['val_ap']:.4f}; device: {device}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
experiments on Drive: ['exp001_siamese_sar_temporal', 'exp001_small_cnn_sar_temporal', 'exp001_tiny_cnn_sar_temporal', 'exp002_small_cnn_sar_temporal']
loaded final pipeline: baseline / xgb_spatial_temporal
validation-fitted threshold: 0.5064
CNN: tiny_cnn, best epoch 5, validation PR-AUC 0.9570; device: cuda


## 2. Score a city/date with the selected final pipeline


In [3]:
def _cnn_scores(d, date, offsets):
    '''CNN probabilities aligned to every canonical building row.'''
    scores = {}
    for offset in offsets:
        score_date = date if offset == 0 else (
            __import__("pipeline").shift_date(date, offset))
        # Offset dates use their temporal composite; date 0 uses the labelled one.
        source, rows = city_source(d, score_date, cfg["features"],
                                   window_days="auto")
        full = np.full(len(d["table"]), np.nan, dtype=np.float32)
        full[rows] = predict_probs(
            model, source, mu=ckpt["mu"], sd=ckpt["sd"], device=device,
            batch=cfg.get("predict_batch", 512))
        scores[offset] = full
    return scores


def _selected_scores(d, date):
    '''Recreate exactly the selected CNN/rule/XGBoost prediction variant.'''
    offsets = temporal_offsets() if (STACKER and STACKER["temporal"]) else [0]
    scores = _cnn_scores(d, date, offsets)
    base = scores[0]
    usable = np.isfinite(base)

    if FINAL_VARIANT == "cnn":
        return base, usable
    if FINAL_VARIANT == "cnn+fixed_neighbor_rule":
        out = np.full_like(base, np.nan)
        sm = cfg["spatial_smoothing"]
        out[usable] = spatial_smooth(
            d["xy"][usable], base[usable], k=sm["k"], weight=sm["weight"])
        return out, usable

    # Whole-city inference intentionally uses neighbouring *unlabelled* buildings.
    # This is suitable for deployment/map display. Notebook 2 keeps neighbourhood
    # features within each split when reporting validation/test metrics.
    spatial = neighbour_features(d["xy"], np.nan_to_num(base, nan=0.0),
                                 ks=tuple(cfg["stacking"]["ks"]))
    features = temporal_features(scores) if STACKER["temporal"] else None
    if features is not None:
        import pandas as pd
        features = pd.concat([spatial, features], axis=1)
    else:
        features = spatial
    out = np.full_like(base, np.nan)
    out[usable] = STACKER["model"].predict_proba(
        features.loc[usable, STACKER["columns"]])[:, 1]
    return out, usable


def predict_city(city, date=None):
    '''Score a labelled assessment date and save an isolated GeoJSON output.'''
    if city not in CITY_REGISTRY:
        raise KeyError(f"unknown city {city!r}; add it to CITY_REGISTRY and run Notebook 1")
    date = date or CITY_REGISTRY[city]["label_dates"][-1]
    if date not in CITY_REGISTRY[city]["label_dates"]:
        raise ValueError(f"{date} is not a labelled assessment date for {city}")

    d = load_city(city)
    score, usable = _selected_scores(d, date)
    rows = np.flatnonzero(usable)
    table = d["table"].iloc[rows].copy()
    label_col = f"class_{date}"
    if label_col not in table:
        raise KeyError(f"{label_col} is absent; Notebook 1 did not create labels for this date")
    labels = label_matrix(d["table"], d["dates"])
    if cfg.get("label_temporal", False):
        labels = propagate_labels(labels)
    date_index = d["dates"].index(date)

    out = table[["geometry"]].copy()
    out["prob"] = score[rows].astype(float)
    out["pred"] = (score[rows] >= FINAL_THRESHOLD).astype(int)
    out["class"] = labels[rows, date_index].astype(int)
    max_change_col = f"max_change_{date}"
    out["max_change"] = (table[max_change_col].to_numpy(float)
                         if max_change_col in table else np.nan)
    out["split"] = split_assignment(
        d["lat"], city, strip_dates(cfg["split"]))[rows]
    out["city"], out["date"] = city, date
    out["pipeline"] = f"{FINAL_TAG} / {FINAL_VARIANT}"
    out["threshold"] = FINAL_THRESHOLD

    safe_variant = FINAL_VARIANT.replace("+", "_")
    path = os.path.join(
        OUT["predictions"], f"{city}_{date}_{FINAL_TAG}_{safe_variant}.geojson")
    out.to_file(path, driver="GeoJSON")
    print(f"wrote {len(out):,} usable buildings to {path}")
    return out, path


# Change these two values, then run this cell.
CITY = "Gaza"
DATE = "20240906"  # None selects the latest labelled date for CITY

predictions, prediction_path = predict_city(CITY, DATE)
predictions.head(3)


/content/drive/MyDrive/War-Damage-Detection/pipeline.py:896: RuntimeWarning: Mean of empty slice
  feats[f"{prefix}_past_mean"] = np.nanmean(S[:, past], axis=1)
/content/drive/MyDrive/War-Damage-Detection/pipeline.py:897: RuntimeWarning: All-NaN slice encountered
  feats[f"{prefix}_past_max"] = np.nanmax(S[:, past], axis=1)
/content/drive/MyDrive/War-Damage-Detection/pipeline.py:900: RuntimeWarning: Mean of empty slice
  feats[f"{prefix}_future_mean"] = np.nanmean(S[:, future], axis=1)
/content/drive/MyDrive/War-Damage-Detection/pipeline.py:901: RuntimeWarning: All-NaN slice encountered
  feats[f"{prefix}_future_min"] = np.nanmin(S[:, future], axis=1)
/content/drive/MyDrive/War-Damage-Detection/pipeline.py:903: RuntimeWarning: Mean of empty slice
  feats[f"{prefix}_mean"] = np.nanmean(S, axis=1)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/c

wrote 214,225 usable buildings to /content/drive/MyDrive/War-Damage-Detection/experiments/exp001_tiny_cnn_sar_temporal/predictions/Gaza_20240906_baseline_xgb_spatial_temporal.geojson


,geometry,prob,pred,class,max_change,split,city,date,pipeline,threshold
0,"POLYGON ((34.44076 31.22094, 34.44076 31.221, ...",0.037626,0,0,NaN,test,Gaza,20240906,baseline / xgb_spatial_temporal,0.506368
2,"POLYGON ((34.42633 31.2222, 34.42614 31.2222, ...",0.108922,0,0,NaN,test,Gaza,20240906,baseline / xgb_spatial_temporal,0.506368
3,"POLYGON ((34.35608 31.22323, 34.35601 31.22352...",0.039365,0,0,NaN,test,Gaza,20240906,baseline / xgb_spatial_temporal,0.506368


## 3. Interactive vector map

The map is a diagnostic display, not another evaluation step. Do not use
test-map errors to alter the chosen model, features, or threshold. For a
new experiment, make those choices on validation data in Notebook 2.


In [4]:
import folium
import matplotlib


def prob_to_color(p):
    return matplotlib.colors.to_hex(matplotlib.colormaps["RdYlGn_r"](float(p)))


def build_comparison_map(pred_gdf, max_features=10_000):
    '''Toggleable truth, confidence, and FP/FN layers on a map.'''
    g = pred_gdf.copy()
    if max_features is not None and len(g) > max_features:
        g = g.sample(max_features, random_state=0)
        print(f"showing a reproducible {len(g):,}-building sample")
    g["color_pred"] = [prob_to_color(p) for p in g["prob"]]
    g["color_true"] = np.where(g["class"] == 1, "#C0392B", "#4C72B0")
    g["error_type"] = np.select(
        [(g["pred"] == 1) & (g["class"] == 0),
         (g["pred"] == 0) & (g["class"] == 1)],
        ["false positive", "false negative"], default="correct")

    center = [g.geometry.centroid.y.mean(), g.geometry.centroid.x.mean()]
    m = folium.Map(location=center, zoom_start=14, tiles=None)
    folium.TileLayer(
        tiles=("https://server.arcgisonline.com/ArcGIS/rest/services/"
               "World_Imagery/MapServer/tile/{z}/{y}/{x}"),
        attr="Esri World Imagery (undated; orientation only)",
        name="satellite basemap").add_to(m)

    fields = ["prob", "class", "pred", "error_type", "max_change", "split", "date"]
    aliases = ["damage probability", "UNOSAT label", "prediction", "outcome",
               "PWTT max_change", "development split", "assessment date"]

    def add_layer(frame, color_col, name, show, weight=1, fill=0.7, outline=None):
        folium.GeoJson(
            frame[["geometry"] + fields + [color_col]].to_json(), name=name, show=show,
            style_function=lambda f, cc=color_col, w=weight, fo=fill, ol=outline: {
                "color": ol or f["properties"][cc], "fillColor": f["properties"][cc],
                "weight": w, "fillOpacity": fo},
            tooltip=folium.GeoJsonTooltip(fields=fields, aliases=aliases),
        ).add_to(m)

    add_layer(g, "color_true", "UNOSAT ground truth", show=False)
    add_layer(g, "color_pred", "selected-pipeline confidence", show=True)
    wrong = g[g["error_type"] != "correct"]
    if len(wrong):
        add_layer(wrong, "color_pred", "false positives / negatives", show=True,
                  weight=3, fill=0.0, outline="#00FFFF")
    folium.LayerControl(collapsed=False).add_to(m)
    return m


build_comparison_map(predictions)


Output hidden; open in https://colab.research.google.com to view.

## 4. Lonboard map — fast vectors with building-level attributes

Lonboard renders the footprints on the GPU and is preferable to Folium for
a large city. Cyan outlines mark false positives and false negatives.


In [5]:
!pip install -q lonboard


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 117.4 MB/s eta 0:00:00


In [6]:
import matplotlib as mpl
from lonboard import Map as LonboardMap, PolygonLayer
from lonboard.colormap import apply_continuous_cmap

g = predictions.to_crs(4326).copy()
fill = apply_continuous_cmap(
    g["prob"].to_numpy(), mpl.colormaps["RdYlGn_r"], alpha=0.7)
line = np.where(
    (g["pred"] != g["class"]).to_numpy()[:, None],
    [0, 255, 255], [0, 0, 0]).astype("uint8")

lonboard_layer = PolygonLayer.from_geopandas(
    g[["geometry", "prob", "class", "pred", "max_change", "split", "date"]],
    get_fill_color=fill,
    get_line_color=line,
    line_width_min_pixels=0.5,
)
LonboardMap(lonboard_layer)


## 5. Earth Engine whole-city map

This option rasterizes the predictions, which is much lighter than drawing
every footprint. It also recreates the pre/post Sentinel composites using
the current preprocessing windows and the **pinned Sentinel-1 orbit** saved
by Notebook 1. Radar is therefore the date-matched model evidence when S1
was enabled. True-colour and infrared Sentinel-2 layers are contextual
evidence only unless S2 was enabled in the experiment.


In [7]:
!pip install -q geemap ipywidgets


In [8]:
import base64
import io
import ee
import geemap
import ipywidgets as widgets
import matplotlib as mpl
import rasterio
from PIL import Image
from ipyleaflet import ImageOverlay
from rasterio.features import rasterize
from rasterio.transform import from_origin
from rasterio.warp import (Resampling, calculate_default_transform,
                           reproject, transform_bounds)

from pipeline import (PREP, pre_raster_path, post_raster_path,
                      read_export_meta)

try:
    ee.Initialize(project=PREP["imagery"]["gee_project"])
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PREP["imagery"]["gee_project"])


def rasterize_predictions(frame, out_path, value="prob", scale_m=10):
    '''Paint one per-building value onto a compact local GeoTIFF.'''
    g = frame.to_crs(frame.estimate_utm_crs()).reset_index(drop=True)
    minx, miny, maxx, maxy = g.total_bounds
    width = int(np.ceil((maxx - minx) / scale_m)) + 1
    height = int(np.ceil((maxy - miny) / scale_m)) + 1
    transform = from_origin(minx, maxy, scale_m, scale_m)

    values = g[value].to_numpy(float)
    # Burn large polygons first so small buildings are not hidden by them.
    order = np.argsort(-g.geometry.area.to_numpy())
    shapes = [(g.geometry.iloc[i], float(values[i])) for i in order
              if np.isfinite(values[i])]
    array = rasterize(shapes, out_shape=(height, width), transform=transform,
                      fill=np.nan, all_touched=True, dtype="float32")
    with rasterio.open(
            out_path, "w", driver="GTiff", height=height, width=width,
            count=1, dtype="float32", crs=g.crs, transform=transform,
            nodata=np.nan, tiled=True, blockxsize=256, blockysize=256,
            compress="deflate") as dst:
        dst.write(array, 1)
    print(f"{len(g):,} buildings -> {width}x{height} raster: {out_path}")
    return out_path


raster_frame = predictions.copy()
MAP_CITY = str(raster_frame["city"].iloc[0])
MAP_DATE = str(raster_frame["date"].iloc[0])
# NaN keeps correct buildings transparent in the disagreement overlay.
raster_frame["error"] = np.where(
    raster_frame["pred"] != raster_frame["class"], 1.0, np.nan)
prediction_stem = os.path.splitext(os.path.basename(prediction_path))[0]
prob_tif = rasterize_predictions(
    raster_frame, os.path.join(OUT["predictions"], prediction_stem + "_prob.tif"),
    "prob")
error_tif = rasterize_predictions(
    raster_frame, os.path.join(OUT["predictions"], prediction_stem + "_error.tif"),
    "error")


214,225 buildings -> 3372x4191 raster: /content/drive/MyDrive/War-Damage-Detection/experiments/exp001_tiny_cnn_sar_temporal/predictions/Gaza_20240906_baseline_xgb_spatial_temporal_prob.tif
214,225 buildings -> 3372x4191 raster: /content/drive/MyDrive/War-Damage-Detection/experiments/exp001_tiny_cnn_sar_temporal/predictions/Gaza_20240906_baseline_xgb_spatial_temporal_error.tif


In [9]:
def raster_aoi(city, date, sensor="s1"):
    '''Use the exact bounds of the post raster consumed by preprocessing.'''
    path = post_raster_path(city, date, sensor)
    if not os.path.exists(path):
        raise FileNotFoundError(f"model-input raster is missing: {path}")
    with rasterio.open(path) as src:
        bounds, crs = src.bounds, src.crs
    if crs.to_epsg() != 4326:
        bounds = transform_bounds(crs, "EPSG:4326", *bounds)
    return ee.Geometry.Rectangle(list(bounds))


def imagery_windows(city, date):
    '''The same labelled pre/post windows used by Notebook 1.'''
    im = PREP["imagery"]
    war = ee.Date(CITY_REGISTRY[city]["war_start"])
    label = ee.Date(f"{date[:4]}-{date[4:6]}-{date[6:8]}")
    pre = (war.advance(-im["pre_months"], "month"), war)
    if im["post_direction"] == "forward":
        post = (label, label.advance(im["post_months"], "month"))
    else:
        post = (label.advance(-im["post_months"], "month"), label)
    return pre, post


def s1_composites(city, aoi, pre, post):
    '''Median radar composites on Notebook 1's pinned relative orbit.'''
    def mask_edges(image):
        valid = image.select("VV").gt(-35).And(image.select("VH").gt(-35))
        return image.updateMask(valid)

    meta = read_export_meta(city)
    if "orbit" not in meta:
        raise RuntimeError(
            f"No pinned orbit in the export metadata for {city}. "
            "Run Notebook 1's orbit-selection/export section first.")
    orbit = int(meta["orbit"])
    collection = (
        ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains(
            "transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.listContains(
            "transmitterReceiverPolarisation", "VH"))
        .filter(ee.Filter.eq("relativeOrbitNumber_start", orbit))
        .select(["VV", "VH"]).map(mask_edges))

    pre_count = collection.filterDate(*pre).size().getInfo()
    post_count = collection.filterDate(*post).size().getInfo()
    if pre_count == 0 or post_count == 0:
        raise RuntimeError(
            f"orbit {orbit} has {pre_count} pre and {post_count} post scenes")
    print(f"Sentinel-1 orbit {orbit}: {pre_count} pre, {post_count} post scenes")

    def radar_rgb(window):
        composite = collection.filterDate(*window).median()
        ratio = composite.select("VV").subtract(
            composite.select("VH")).rename("ratio")
        return composite.addBands(ratio).select(
            ["VV", "VH", "ratio"]).clip(aoi)

    return radar_rgb(pre), radar_rgb(post)


def s2_composites(aoi, pre, post, max_cloud=None):
    '''Cloud-masked Sentinel-2 context for the same two windows.'''
    max_cloud = max_cloud or PREP["imagery"]["s2_max_cloud"]

    def mask_scl(image):
        scl = image.select("SCL")
        bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10))
        return image.updateMask(bad.Not())

    def mask_qa60(image):
        qa = image.select("QA60")
        clear = qa.bitwiseAnd(1 << 10).eq(0).And(
            qa.bitwiseAnd(1 << 11).eq(0))
        return image.updateMask(clear)

    def composite(window, tag):
        choices = [("COPERNICUS/S2_SR_HARMONIZED", mask_scl, "SR"),
                   ("COPERNICUS/S2_HARMONIZED", mask_qa60, "TOA")]
        for collection_id, mask, level in choices:
            collection = (
                ee.ImageCollection(collection_id).filterBounds(aoi)
                .filterDate(*window)
                .filter(ee.Filter.lt(
                    "CLOUDY_PIXEL_PERCENTAGE", max_cloud)))
            count = collection.size().getInfo()
            if count:
                print(f"Sentinel-2 {tag}: {count} scenes ({level})")
                return collection.map(mask).median().clip(aoi)
        raise RuntimeError(f"no usable Sentinel-2 scenes in {tag} window")

    return composite(pre, "pre-war"), composite(post, "post-event")


aoi = raster_aoi(MAP_CITY, MAP_DATE)
pre_window, post_window = imagery_windows(MAP_CITY, MAP_DATE)
s1_pre, s1_post = s1_composites(
    MAP_CITY, aoi, pre_window, post_window)

VIS = {
    "radar": ({"min": [-20, -27, 3], "max": [0, -10, 14]},
              s1_pre, s1_post),
}
# Optical imagery is useful context but is not required for the radar map.
try:
    s2_pre, s2_post = s2_composites(aoi, pre_window, post_window)
    VIS.update({
        "true colour": ({"bands": ["B4", "B3", "B2"], "min": 0,
                         "max": 3000, "gamma": 1.15}, s2_pre, s2_post),
        "infrared": ({"bands": ["B8", "B4", "B3"], "min": 0,
                      "max": 4000, "gamma": 1.15}, s2_pre, s2_post),
    })
except Exception as exc:
    print(f"Sentinel-2 context unavailable; continuing with radar: {exc}")
imagery_layers = {}
for view, (vis, pre_image, post_image) in VIS.items():
    for when, image in [("pre-war", pre_image), ("post-event", post_image)]:
        imagery_layers[(when, view)] = geemap.ee_tile_layer(
            image, vis, f"{when} {view}")


Sentinel-1 orbit 94: 30 pre, 3 post scenes
Sentinel-2 pre-war: 127 scenes (SR)
Sentinel-2 post-event: 8 scenes (SR)


In [10]:
def add_raster_overlay(map_widget, tif, name, cmap="RdYlGn_r",
                       vmin=0, vmax=1, opacity=0.75, max_px=4000):
    '''Embed a local raster as one transparent PNG layer.'''
    with rasterio.open(tif) as src:
        transform, width, height = calculate_default_transform(
            src.crs, "EPSG:4326", src.width, src.height, *src.bounds)
        if max(width, height) > max_px:
            factor = max_px / max(width, height)
            transform, width, height = calculate_default_transform(
                src.crs, "EPSG:4326", src.width, src.height, *src.bounds,
                dst_width=max(1, int(width * factor)),
                dst_height=max(1, int(height * factor)))
        array = np.full((height, width), np.nan, np.float32)
        reproject(
            rasterio.band(src, 1), array, src_transform=src.transform,
            src_crs=src.crs, dst_transform=transform, dst_crs="EPSG:4326",
            resampling=Resampling.nearest, dst_nodata=np.nan)

    west, north = transform * (0, 0)
    east, south = transform * (width, height)
    normalized = np.clip((array - vmin) / (vmax - vmin), 0, 1)
    rgba = (mpl.colormaps[cmap](np.nan_to_num(normalized)) * 255).astype(np.uint8)
    rgba[..., 3] = np.where(np.isfinite(array), int(opacity * 255), 0)
    buffer = io.BytesIO()
    Image.fromarray(rgba, mode="RGBA").save(
        buffer, format="PNG", optimize=True)
    url = "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode()
    print(f"{name}: {len(buffer.getvalue()) / 1e6:.1f} MB PNG")
    map_widget.add_layer(ImageOverlay(
        url=url, bounds=((south, west), (north, east)), name=name))


def imagery_switcher(layers, default=("post-event", "radar")):
    '''Keep exactly one dated imagery layer visible.'''
    whens = ["pre-war", "post-event"]
    views = list(dict.fromkeys(key[1] for key in layers))
    when_button = widgets.ToggleButtons(
        options=whens, value=default[0], description="date:")
    view_button = widgets.ToggleButtons(
        options=views, value=default[1], description="view:")

    def refresh(*_):
        selected = (when_button.value, view_button.value)
        for key, layer in layers.items():
            layer.visible = key == selected

    when_button.observe(refresh, names="value")
    view_button.observe(refresh, names="value")
    refresh()
    return widgets.VBox([when_button, view_button])


GEE_MAP = geemap.Map()
GEE_MAP.add_basemap("SATELLITE")
GEE_MAP.centerObject(aoi, 12)
for layer in imagery_layers.values():
    GEE_MAP.add_layer(layer, {}, layer.name, False)
add_raster_overlay(GEE_MAP, prob_tif, "selected-pipeline probability")
add_raster_overlay(GEE_MAP, error_tif, "FP/FN", cmap="cool_r", opacity=0.95)

controls = imagery_switcher(imagery_layers)
widgets.VBox([controls, GEE_MAP])


selected-pipeline probability: 2.5 MB PNG
FP/FN: 0.3 MB PNG


### Earth Engine before/after slider

Choose `radar`, `true colour`, or `infrared`, then drag the divider.


In [12]:
VIEW = "true colour"  # "radar", "true colour", or "infrared"
vis, pre_image, post_image = VIS[VIEW]

split_map = geemap.Map()
split_map.centerObject(aoi, 13)
split_map.split_map(
    left_layer=geemap.ee_tile_layer(pre_image, vis, f"pre-war {VIEW}"),
    right_layer=geemap.ee_tile_layer(post_image, vis, f"post-event {VIEW}"))
split_map


Map(center=[31.409096415747218, 34.39249472592175], controls=(ZoomControl(options=['position', 'zoom_in_text',…